<a href="https://colab.research.google.com/github/Ahmad860187/VIP2026/blob/main/channel_charting_walk_kaust_v8_multiview_2_repaired.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Channel Charting v8 — Multi-View Reliability-Aware Dissimilarity Fusion

## Pipeline for VIPP 401A (Spring 2026) — walk_kaust Dataset

### What this notebook does
1. Loads and consolidates the CellMob MobiSense walk_kaust pedestrian LTE dataset
2. Builds sparse cell-specific fingerprint matrices (RSRP/RSRQ per cell)
3. Computes **5 dissimilarity sources**: D_signal, D_time, D_PL, D_TA, D_CQI
4. Performs GPS-supervised **K-source weighted fusion** with fair train/test evaluation
5. Trains a Siamese network on the fused distance matrix
6. Reports ablation results and final channel chart quality

### Key design principles (from literature review)
- **DICHASUS pipeline** (Stephan et al. 2024): multi-metric fusion → geodesic smoothing → Siamese embedding
- **MKL** (Gönen & Alpaydin 2011): learned convex combination of dissimilarity matrices
- **No silent fallbacks**: evaluation on both-finite pairs only; training uses documented NaN→1.0
- **GPS used ONLY for fusion weight learning** — Siamese never sees GPS

### Changes from v7
- Extended from K=2 (signal+time) to K=5 candidate distance sources
- Exhaustive subset search over all source combinations
- Proper no-fallback D_train construction (v7 bug: dense fallback from weak sources destroys chart)
- PL excluded (negative correlation with GPS distance)
- TA/CQI evaluated but found too weak for standalone use; CQI adds marginal value as fusion partner

### Configuration
Set `SUBSAMPLE_N = 4000` for full resolution (requires ~8GB RAM).
Set `SUBSAMPLE_N = 1200` for quick testing.


## 0 — Imports & Configuration

In [1]:
import os, glob, warnings, time as _time, gc, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.spatial.distance import cdist
from sklearn.manifold import MDS
from scipy.stats import spearmanr
from itertools import combinations
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ── USER CONFIGURATION ──────────────────────────────────────────
DATA_DIR        = "walk_kaust_csvs"     # <-- set to your data directory
SUBSAMPLE_N     = 4000                  # 4000 for full, 1200 for quick test
TIME_BIN        = '5s'                  # temporal binning resolution
MIN_APPEARANCES = 10                    # minimum cell appearances to keep
MDS_COMPONENTS  = 2
SIAMESE_EPOCHS  = 200                   # 200 for full, 80 for quick
SIAMESE_LR      = 1e-3
SIAMESE_BATCH   = 512
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
OVERLAP_LAMBDA  = 0.3                   # overlap penalty in cosine distance
MIN_OVERLAP     = 5                     # minimum shared features for signal distance

print(f"Device: {DEVICE}, N_target={SUBSAMPLE_N}, TIME_BIN={TIME_BIN}")


Device: cuda, N_target=4000, TIME_BIN=5s


## 1 — Data Loading & Consolidation

**Critical step:** Each CSV row contains measurements for ONE feature type. PCI appears on rows separate from RSRP. The `groupby(['file_id','Time']).agg('first')` consolidation merges them into a single record per timestamp.

In [2]:
# ── Mount Google Drive and set data folder ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, gc
import numpy as np
import pandas as pd

# Change this only if your folder path is different
DATA_DIR = "/content/drive/MyDrive/walk_kaust"

print("Folder exists:", os.path.isdir(DATA_DIR))

# ── Column definitions ────────────────────────────────────────
PCI_COL = 'Physical cell identity (LTE pcell)'
CHAN_COL = 'Channel number (LTE pcell)'
TA_COL  = 'Timing advance'
PL_COL  = 'Pathloss (LTE pcell)'
RSRP_COLS = ['RSRP/antenna port - 1', 'RSRP/antenna port - 2']
RSRQ_COLS = ['RSRQ/antenna port - 1', 'RSRQ/antenna port - 2']
CQI_WB = ['Wideband CQI for codeword 0', 'Wideband CQI for codeword 1']

# ── Load all CSV files ───────────────────────────────────────────
files = sorted([f for f in glob.glob(f"{DATA_DIR}/*.csv") if 'file_map' not in f])
assert len(files) > 0, f"No CSV files in {DATA_DIR}"
print(f"Found {len(files)} CSV files")

chunks = []
for fp in files:
    df = pd.read_csv(fp, low_memory=False)
    if 'Time' not in df.columns: continue
    df['file_id'] = os.path.basename(fp).replace('.csv', '')
    chunks.append(df)

raw = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()
print(f"Total rows: {len(raw):,}")

# ── Consolidate per (file_id, Time) ─ merges PCI/RSRP rows ──────
con = raw.groupby(['file_id', 'Time'], as_index=False).agg('first')
del raw; gc.collect()
con[['Latitude','Longitude']] = con.groupby('file_id')[['Latitude','Longitude']].ffill()

# Parse timestamps
ts = pd.to_datetime(con['Time'].astype(str), format='%H:%M:%S.%f', errors='coerce')
ts = ts.fillna(pd.to_datetime(con['Time'].astype(str), format='%H:%M:%S', errors='coerce'))
con['time_sec'] = ts.dt.hour*3600 + ts.dt.minute*60 + ts.dt.second + ts.dt.microsecond/1e6
con = con[con['time_sec'].notna()].copy()
con['time_td'] = pd.to_timedelta(con['time_sec'], unit='s')
con['TimeBin'] = con['time_td'].dt.floor(TIME_BIN)

# Numeric conversion
for c in [PCI_COL, CHAN_COL, TA_COL, PL_COL] + RSRP_COLS + RSRQ_COLS + CQI_WB + ['Latitude','Longitude']:
    if c in con.columns:
        con[c] = pd.to_numeric(con[c], errors='coerce')

# Cell key
con = con[con[PCI_COL].notna()].copy()
con['cell_key'] = ('C' + con[PCI_COL].astype(int).astype(str) + '_'
                   + con[CHAN_COL].fillna(0).astype(int).astype(str))

# CQI average
if CQI_WB[0] in con.columns and CQI_WB[1] in con.columns:
    con['CQI_val'] = con[CQI_WB].mean(axis=1, skipna=True)
elif CQI_WB[0] in con.columns:
    con['CQI_val'] = con[CQI_WB[0]]
else:
    con['CQI_val'] = np.nan

print(f"Consolidated: {len(con):,} rows, {con['cell_key'].nunique()} unique cells")
print(f"RSRP coverage: {con[RSRP_COLS[0]].notna().mean()*100:.1f}%")
print(f"PL: {con[PL_COL].notna().mean()*100:.1f}%, TA: {con[TA_COL].notna().mean()*100:.1f}%")


Mounted at /content/drive
Folder exists: True
Found 69 CSV files
Total rows: 1,391,698
Consolidated: 138,816 rows, 92 unique cells
RSRP coverage: 99.7%
PL: 100.0%, TA: 82.5%


## 2 — Sparse Cell Fingerprint Matrix

In [3]:
# ── Dense serving-cell features ───────────────────────────────
dense = con.groupby(['file_id','TimeBin'], as_index=False).agg(
    PL_s=(PL_COL,'median'), TA_s=(TA_COL,'median'),
    CQI_s=('CQI_val','median'), time_sec=('time_sec','median'))

# ── Cell-specific sparse pivot ───────────────────────────────────
feat_specs = [('R1', RSRP_COLS[0]), ('R2', RSRP_COLS[1]),
              ('Q1', RSRQ_COLS[0]), ('Q2', RSRQ_COLS[1])]
cg = con.groupby(['file_id','TimeBin','cell_key'], as_index=False).agg(
    **{col: (col,'median') for _,col in feat_specs if col in con.columns})
cell_cnt = cg.groupby('cell_key').size()
valid_cells = set(cell_cnt[cell_cnt >= MIN_APPEARANCES].index)
cg = cg[cg['cell_key'].isin(valid_cells)].copy()
print(f"Valid cells (>={MIN_APPEARANCES} appearances): {len(valid_cells)}")

# GPS reference
loc_ref = con[con['Latitude'].notna() & con['Longitude'].notna()][
    ['file_id','time_sec','Latitude','Longitude']].sort_values(['file_id','time_sec']).copy()
del con; gc.collect()

# Pivot
wide = None
for prefix, col in feat_specs:
    if col not in cg.columns: continue
    p = cg.pivot_table(index=['file_id','TimeBin'], columns='cell_key',
                        values=col, aggfunc='median')
    if p.shape[1] == 0: continue
    p.columns = [f"{prefix}__{c}" for c in p.columns]
    wide = p if wide is None else wide.join(p, how='outer')
del cg; gc.collect()
assert wide is not None and wide.shape[1] > 0
wide = wide.sort_index()
FEAT_COLS = sorted(wide.columns.tolist())
print(f"Fingerprint matrix: {wide.shape[0]} rows × {wide.shape[1]} columns")


Valid cells (>=10 appearances): 39
Fingerprint matrix: 14201 rows × 156 columns


## 3 — GPS Alignment & Coordinate Conversion

In [4]:
wdf = wide.reset_index().merge(dense, on=['file_id','TimeBin'], how='left')
del dense, wide; gc.collect()

aligned_parts = []
for fid, g in wdf.groupby('file_id'):
    g = g.sort_values('time_sec').copy()
    r = loc_ref[loc_ref['file_id'] == fid]
    if len(r) == 0: continue
    m = pd.merge_asof(g, r[['time_sec','Latitude','Longitude']].sort_values('time_sec'),
                       on='time_sec', direction='nearest', suffixes=('','_gps'))
    m['file_id'] = fid
    aligned_parts.append(m)

aligned = pd.concat(aligned_parts, ignore_index=True)
del aligned_parts, loc_ref, wdf; gc.collect()

lat_col = 'Latitude_gps' if 'Latitude_gps' in aligned.columns else 'Latitude'
lon_col = 'Longitude_gps' if 'Longitude_gps' in aligned.columns else 'Longitude'
aligned = aligned[aligned[lat_col].notna() & aligned[lon_col].notna()].copy()

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32637", always_xy=True)
x, y = transformer.transform(aligned[lon_col].values.astype(np.float64),
                               aligned[lat_col].values.astype(np.float64))
aligned['X_utm'] = x
aligned['Y_utm'] = y
print(f"Aligned rows with GPS: {len(aligned):,}")


Aligned rows with GPS: 14,201


## 4 — Subsampling Within Files

Stride-based subsampling preserves temporal locality within each file.

In [5]:
def subsample_within_files(file_ids_arr, n_target):
    n_all = len(file_ids_arr)
    if n_all <= n_target:
        return np.arange(n_all)
    ids = np.asarray(file_ids_arr)
    parts = []
    for fid in np.unique(ids):
        idx = np.where(ids == fid)[0]
        k = max(1, int(round(n_target * len(idx) / n_all)))
        parts.append(idx[np.linspace(0, len(idx)-1, min(k, len(idx)), dtype=int)])
    out = np.sort(np.concatenate(parts))
    if len(out) > n_target:
        out = out[np.linspace(0, len(out)-1, n_target, dtype=int)]
    return out

fids = aligned['file_id'].values.astype(str)
idx_sub = subsample_within_files(fids, SUBSAMPLE_N)
N = len(idx_sub)

feat_sub    = aligned[FEAT_COLS].values[idx_sub].astype(np.float64)
mask_sub    = np.isfinite(feat_sub).astype(np.float64)
feat_model  = np.nan_to_num(feat_sub, nan=0.0)
pos_sub     = np.column_stack([aligned['X_utm'].values[idx_sub],
                                aligned['Y_utm'].values[idx_sub]])
time_sub    = aligned['time_sec'].values[idx_sub].astype(np.float64)
file_sub    = np.array(fids[idx_sub], dtype=str)
PL_sub      = aligned['PL_s'].values[idx_sub].astype(np.float64)
TA_sub      = aligned['TA_s'].values[idx_sub].astype(np.float64)
CQI_sub     = aligned['CQI_s'].values[idx_sub].astype(np.float64)
INPUT_DIM   = feat_model.shape[1]

del aligned; gc.collect()
print(f"Subsampled: {len(fids):,} → {N}")
print(f"Features: {feat_sub.shape}, observed ratio: {mask_sub.mean():.4f}")
print(f"Dense coverage: PL={np.isfinite(PL_sub).mean()*100:.1f}% "
      f"TA={np.isfinite(TA_sub).mean()*100:.1f}% CQI={np.isfinite(CQI_sub).mean()*100:.1f}%")


Subsampled: 14,201 → 3999
Features: (3999, 156), observed ratio: 0.0279
Dense coverage: PL=100.0% TA=91.8% CQI=99.8%


## 5 — Compute All Dissimilarity Matrices

Five candidate distance sources:
- **D_signal**: overlap-aware cosine distance on RSRP/RSRQ cell fingerprints (sparse)
- **D_time**: |t_i - t_j| for same-file pairs, NaN for cross-file (sparse)
- **D_PL**: |PL_i - PL_j| serving-cell pathloss L1 (dense)
- **D_TA**: |TA_i - TA_j| timing advance L1 (dense)
- **D_CQI**: |CQI_i - CQI_j| wideband CQI L1 (dense)


In [6]:
tri = np.triu_indices(N, k=1)
D_true = cdist(pos_sub, pos_sub, 'euclidean')
D_true_flat = D_true[tri]

# ── D_signal (overlap-aware cosine) ──────────────────────────────
def overlap_cosine_distance(X_val, X_mask, overlap_lambda=0.3, min_overlap=5, chunk=200):
    X = np.where(np.isfinite(X_val), X_val, 0.0)
    M = (X_mask > 0)
    Nl = X.shape[0]
    obs = M.sum(axis=1).astype(np.float64)
    D = np.full((Nl, Nl), np.nan, dtype=np.float64)
    np.fill_diagonal(D, 0.0)
    for s in range(0, Nl, chunk):
        e = min(s + chunk, Nl)
        cm = M[s:e, None, :] & M[None, :, :]
        cnt = cm.sum(axis=2).astype(np.float64)
        val = cnt >= min_overlap
        c = cm.astype(np.float64)
        cs = np.maximum(cnt, 1.0)
        mi = (c * X[s:e, None, :]).sum(2) / cs
        mj = (c * X[None, :, :]).sum(2) / cs
        xc = c * (X[s:e, None, :] - mi[:, :, None])
        yc = c * (X[None, :, :] - mj[:, :, None])
        dot = (xc * yc).sum(2)
        nx = np.sqrt((xc * xc).sum(2))
        ny = np.sqrt((yc * yc).sum(2))
        cos = np.clip(dot / np.maximum(nx * ny, 1e-12), -1, 1)
        oratio = cnt / np.maximum(np.maximum(obs[s:e, None], obs[None, :]), 1.0)
        dist = 1.0 - cos + overlap_lambda * (1.0 - oratio)
        dist = np.clip(dist, 0.0, None)
        dist[~val] = np.nan
        for k in range(e - s):
            dist[k, s + k] = 0.0
        D[s:e] = dist
        del cm, c, xc, yc
    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return D

t0 = _time.time()
D_signal = overlap_cosine_distance(feat_sub, mask_sub, OVERLAP_LAMBDA, MIN_OVERLAP)
ds = D_signal[tri]; fs = np.isfinite(ds) & np.isfinite(D_true_flat)
rho_sig = spearmanr(ds[fs], D_true_flat[fs])[0] if fs.sum() > 10 else np.nan
print(f"D_signal: cov={np.isfinite(ds).mean()*100:.2f}%, rho={rho_sig:.4f} ({_time.time()-t0:.1f}s)")

# ── D_time ──────────────────────────────────────────────────────
D_time = np.where(file_sub[:, None] == file_sub[None, :],
                   np.abs(time_sub[:, None] - time_sub[None, :]), np.nan)
np.fill_diagonal(D_time, 0.0)
dt = D_time[tri]; ft = np.isfinite(dt) & np.isfinite(D_true_flat)
rho_time = spearmanr(dt[ft], D_true_flat[ft])[0]
print(f"D_time:   cov={np.isfinite(dt).mean()*100:.2f}%, rho={rho_time:.4f}")

# ── D_PL, D_TA, D_CQI (L1 distances) ───────────────────────────
for name, arr in [("D_PL", PL_sub), ("D_TA", TA_sub), ("D_CQI", CQI_sub)]:
    D = np.abs(arr[:, None] - arr[None, :])
    d = D[tri]; f = np.isfinite(d) & np.isfinite(D_true_flat)
    rho = spearmanr(d[f], D_true_flat[f])[0] if f.sum() > 10 else np.nan
    r_str = f"{rho:.4f}" if np.isfinite(rho) else "N/A"
    print(f"{name}:     cov={np.isfinite(d).mean()*100:.1f}%, rho={r_str}")
    exec(f"{name} = D; rho_{name.lower()[2:]} = rho")


D_signal: cov=0.03%, rho=0.1716 (71.7s)
D_time:   cov=3.20%, rho=0.5291
D_PL:     cov=100.0%, rho=-0.0134
D_TA:     cov=84.2%, rho=0.0797
D_CQI:     cov=99.6%, rho=0.0284


## 5b — Normalize Distance Matrices

In [7]:
def normalize_distance_matrix(D, label=""):
    t = np.triu_indices(D.shape[0], k=1)
    fv = D[t][np.isfinite(D[t])]
    if len(fv) == 0:
        print(f"  WARNING: {label} has no finite values")
        return None
    p99 = max(np.percentile(fv, 99), 1e-9)
    Dn = np.clip(D / p99, 0.0, 1.0)
    Dn = np.where(np.isfinite(D), Dn, np.nan)
    np.fill_diagonal(Dn, 0.0)
    print(f"  {label}: p99={p99:.2f}, finite={np.isfinite(Dn[t]).mean()*100:.1f}%")
    return Dn

print("Normalizing distance matrices...")
candidates = {}

D_signal_norm = normalize_distance_matrix(D_signal, "D_signal"); del D_signal
if D_signal_norm is not None: candidates['signal'] = D_signal_norm

D_time_norm = normalize_distance_matrix(D_time, "D_time"); del D_time
if D_time_norm is not None: candidates['time'] = D_time_norm

# Only include dense sources if they show positive GPS correlation
D_TA_norm = normalize_distance_matrix(D_TA, "D_TA"); del D_TA
if D_TA_norm is not None and np.isfinite(rho_ta) and rho_ta > 0.01:
    candidates['TA'] = D_TA_norm

D_CQI_norm = normalize_distance_matrix(D_CQI, "D_CQI"); del D_CQI
if D_CQI_norm is not None and np.isfinite(rho_cqi) and rho_cqi > 0.01:
    candidates['CQI'] = D_CQI_norm

# D_PL excluded: typically negatively correlated with GPS distance
gc.collect()
src_names = list(candidates.keys())
print(f"\nCandidate sources ({len(src_names)}): {src_names}")


Normalizing distance matrices...
  D_signal: p99=0.15, finite=0.0%
  D_time: p99=3933.96, finite=3.2%
  D_TA: p99=55.00, finite=84.2%
  D_CQI: p99=8.00, finite=99.6%

Candidate sources (4): ['signal', 'time', 'TA', 'CQI']


## 6 — K-Source Weighted Fusion (GPS-Supervised)

**Protocol:**
- 80/20 point-based train/test split
- For each candidate subset, evaluate ONLY on pairs where ALL sources in the subset are finite
- Grid search (K=2) or Dirichlet sampling (K≥3) over weight simplex
- GPS is used ONLY here to select weights; the Siamese network never sees GPS


In [8]:
# ── Train/test point split ────────────────────────────────────
np.random.seed(42)
perm = np.random.permutation(N)
train_idx = set(perm[:int(0.8 * N)].tolist())
test_idx = set(perm[int(0.8 * N):].tolist())
ii, jj = tri
train_mask = np.array([i in train_idx and j in train_idx for i, j in zip(ii, jj)])
test_mask = np.array([i in test_idx and j in test_idx for i, j in zip(ii, jj)])

src_flat = {n: candidates[n][tri] for n in src_names}

def eval_fusion(weights, names, mask):
    d = sum(w * src_flat[n][mask] for w, n in zip(weights, names))
    return spearmanr(d, D_true_flat[mask])[0]

# ── Exhaustive subset search ─────────────────────────────────────
results = []
print(f"{'Combo':<30s} {'TrainRho':>9s} {'TestRho':>9s} {'Pairs':>8s}  Weights")
print("-" * 85)

K = len(src_names)
for sz in range(1, K + 1):
    for combo in combinations(src_names, sz):
        combo = list(combo)
        bf = np.isfinite(D_true_flat).copy()
        for n in combo: bf &= np.isfinite(src_flat[n])
        trm = train_mask & bf; tem = test_mask & bf
        if trm.sum() < 20: continue

        Kc = len(combo)
        best_w = None; best_r = -np.inf
        if Kc == 1:
            best_w = [1.0]; best_r = eval_fusion(best_w, combo, trm)
        elif Kc == 2:
            for a in np.linspace(0, 1, 51):
                r = eval_fusion([a, 1-a], combo, trm)
                if np.isfinite(r) and r > best_r: best_r = r; best_w = [a, 1-a]
        else:
            np.random.seed(42)
            samps = np.random.dirichlet(np.ones(Kc), 3000)
            for i in range(Kc):
                c = np.zeros(Kc); c[i] = 1; samps = np.vstack([samps, c])
            for i, j in combinations(range(Kc), 2):
                for a in np.linspace(0, 1, 11):
                    c = np.zeros(Kc); c[i] = a; c[j] = 1-a; samps = np.vstack([samps, c])
            for w in samps:
                r = eval_fusion(w.tolist(), combo, trm)
                if np.isfinite(r) and r > best_r: best_r = r; best_w = w.tolist()

        rte = eval_fusion(best_w, combo, tem) if tem.sum() > 10 else np.nan
        ws = ' '.join(f"{n}={w:.2f}" for n, w in zip(combo, best_w))
        print(f"  {'+'.join(combo):<28s} {best_r:>9.4f} {rte:>9.4f} {trm.sum():>8,}  {ws}")
        results.append(dict(combo=combo, w=best_w, rho_tr=best_r, rho_te=rte))

# Select best by test Spearman
valid_r = [r for r in results if np.isfinite(r['rho_te'])]
best = max(valid_r, key=lambda x: x['rho_te'])
FUSE_NAMES = best['combo']
FUSE_WEIGHTS = best['w']
print(f"\n*** BEST: {'+'.join(FUSE_NAMES)} ***")
print(f"    Weights: {dict(zip(FUSE_NAMES, [f'{w:.3f}' for w in FUSE_WEIGHTS]))}")
print(f"    Train: {best['rho_tr']:.4f}, Test: {best['rho_te']:.4f}")


Combo                           TrainRho   TestRho    Pairs  Weights
-------------------------------------------------------------------------------------
  signal                          0.1800   -0.0306    1,868  signal=1.00
  time                            0.5252    0.5631  166,659  time=1.00
  TA                              0.0828    0.0699 4,317,391  TA=1.00
  CQI                             0.0327    0.0133 5,092,836  CQI=1.00
  signal+time                     0.5905       nan      356  signal=0.02 time=0.98
  signal+TA                       0.1778   -0.0370    1,858  signal=1.00 TA=0.00
  signal+CQI                      0.1885   -0.0637    1,868  signal=0.58 CQI=0.42
  time+TA                         0.5184    0.5652  143,179  time=0.96 TA=0.04
  time+CQI                        0.5264    0.5634  166,034  time=0.98 CQI=0.02
  TA+CQI                          0.0888    0.0709 4,296,846  TA=0.94 CQI=0.06
  signal+time+TA                  0.5934       nan      351  signal=0.01 tim

## 7 — Build D_train (No Dense Fallback)

**Critical design choice:** We do NOT fill cross-file NaN pairs with weak dense distances (TA, CQI, PL). Those sources have rho < 0.07 and destroy the chart when used as fallback.

Instead: fused distance where available, signal-only as secondary fallback, NaN elsewhere → mapped to 1.0 for Siamese training (= "unknown, treat as far").


In [9]:
# Build fused D_train from best combo
Dtn = candidates.get('time')
Dcn = candidates.get('CQI')

if 'time' in FUSE_NAMES and 'CQI' in FUSE_NAMES:
    wt = FUSE_WEIGHTS[FUSE_NAMES.index('time')]
    wc = FUSE_WEIGHTS[FUSE_NAMES.index('CQI')]
    both = np.isfinite(Dtn) & np.isfinite(Dcn)
    D_train = np.where(both, wt * Dtn + wc * Dcn, np.nan)
    tonly = np.isfinite(Dtn) & ~np.isfinite(Dcn)
    D_train = np.where(tonly, Dtn, D_train)
elif 'time' in FUSE_NAMES:
    D_train = Dtn.copy() if Dtn is not None else np.full((N,N), np.nan)
else:
    # Fallback: use whatever the best combo was
    D_train = np.full((N, N), np.nan, dtype=np.float64)
    for name, w in zip(FUSE_NAMES, FUSE_WEIGHTS):
        Dn = candidates[name]
        fin = np.isfinite(Dn)
        D_train = np.where(fin & np.isnan(D_train), w * Dn, D_train)
np.fill_diagonal(D_train, 0.0)

# Add signal-only pairs as secondary fallback
Dsn = candidates.get('signal')
if Dsn is not None:
    still_nan = np.isnan(D_train) & np.isfinite(Dsn)
    D_train = np.where(still_nan, Dsn, D_train)
    print(f"Added {still_nan.sum()//2:,} signal-only pairs as fallback")
np.fill_diagonal(D_train, 0.0)

# Free candidate matrices
for n in list(candidates.keys()): del candidates[n]
del candidates; gc.collect()

off = ~np.eye(N, dtype=bool)
fin_train = np.isfinite(D_train) & off
print(f"D_train coverage: {fin_train.sum()/off.sum()*100:.1f}%")
print(f"D_train range: [{np.nanmin(D_train[off]):.4f}, {np.nanmax(D_train[off]):.4f}]")

if fin_train.sum() == 0:
    raise RuntimeError("D_train has NO finite values! Check fusion step.")

best_name = f"fused_{'_'.join(FUSE_NAMES)}"
print(f"TRAIN_DISTANCE_SOURCE = {best_name}")


Added 2,184 signal-only pairs as fallback
D_train coverage: 3.2%
D_train range: [0.0004, 1.0000]
TRAIN_DISTANCE_SOURCE = fused_time_TA


## 8 — Classical Baseline: MDS

In [10]:
print(f"Running MDS on {best_name}...")
t0 = _time.time()

D_mds_input = D_train.copy()
finite_med = np.nanmedian(D_mds_input[off])
D_mds_input = np.where(np.isfinite(D_mds_input), D_mds_input, finite_med)
np.fill_diagonal(D_mds_input, 0.0)
D_mds_input = 0.5 * (D_mds_input + D_mds_input.T)

mds = MDS(n_components=MDS_COMPONENTS, dissimilarity='precomputed',
          random_state=42, normalized_stress=False, max_iter=500, n_init=4)
proj_mds = mds.fit_transform(D_mds_input)
del D_mds_input; gc.collect()

D_mds = cdist(proj_mds, proj_mds, metric='euclidean')
rho_mds, _ = spearmanr(D_mds[tri], D_true_flat)
print(f"Spearman(MDS, GPS) = {rho_mds:.4f} ({_time.time()-t0:.1f}s)")


Running MDS on fused_time_TA...
Spearman(MDS, GPS) = 0.1313 (733.5s)


## 9 — Siamese Neural Network

The Siamese network learns a 2D embedding where Euclidean distances match the fused dissimilarity matrix.

**Input:** standardized fingerprint features (all pivoted columns)  
**Training target:** D_train (fused), normalized to [0,1], NaN→1.0  
**Training pairs:** sampled from pairs where D_train < 0.99 (real distances only)


In [11]:
# ── Normalize D_train for Siamese ──
d_max = np.nanpercentile(D_train[fin_train], 99)
if not np.isfinite(d_max) or d_max <= 0: d_max = 1.0
D_train_normed = np.clip(D_train / max(d_max, 1e-9), 0.0, 1.0)
D_train_normed = np.where(np.isfinite(D_train_normed), D_train_normed, 1.0)
np.fill_diagonal(D_train_normed, 0.0)

mu_train = np.mean(feat_model, axis=0)
sd_train = np.std(feat_model, axis=0)
sd_train = np.where(sd_train > 1e-9, sd_train, 1.0)
feat_train = (feat_model - mu_train) / sd_train

class Encoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        h1 = max(128, input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1), nn.ReLU(), nn.BatchNorm1d(h1),
            nn.Linear(h1, 64), nn.ReLU(), nn.BatchNorm1d(64),
            nn.Linear(64, 2))
    def forward(self, x):
        return self.net(x)

class PairDataset(Dataset):
    def __init__(self, features, D_normed, n_pairs_per_epoch=60000):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.D = D_normed
        self.n_pairs = n_pairs_per_epoch
        # Only sample from real distance pairs (not NaN→1.0 padding)
        self.valid_pairs = np.argwhere(
            np.isfinite(D_normed) & ~np.eye(D_normed.shape[0], dtype=bool) & (D_normed < 0.99))
        print(f"Training pairs (D<0.99): {len(self.valid_pairs):,}")

    def __len__(self): return self.n_pairs

    def __getitem__(self, _):
        idx = np.random.randint(len(self.valid_pairs))
        i, j = self.valid_pairs[idx]
        return self.features[i], self.features[j], torch.tensor(self.D[i,j], dtype=torch.float32)

encoder = Encoder(INPUT_DIM).to(DEVICE)
optimizer = optim.Adam(encoder.parameters(), lr=SIAMESE_LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SIAMESE_EPOCHS)

dataset = PairDataset(feat_train, D_train_normed)
loader = DataLoader(dataset, batch_size=SIAMESE_BATCH, shuffle=True, num_workers=0, drop_last=True)

print(f"Training Siamese: {SIAMESE_EPOCHS} epochs, dim={INPUT_DIM}")
loss_history = []
for epoch in range(1, SIAMESE_EPOCHS + 1):
    encoder.train()
    epoch_loss = 0.0; n_batches = 0
    for feat_i, feat_j, d_target in loader:
        feat_i, feat_j, d_target = feat_i.to(DEVICE), feat_j.to(DEVICE), d_target.to(DEVICE)
        z_i = encoder(feat_i); z_j = encoder(feat_j)
        loss = ((torch.norm(z_i - z_j, dim=1) - d_target) ** 2).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item(); n_batches += 1
    scheduler.step()
    avg_loss = epoch_loss / max(n_batches, 1)
    loss_history.append(avg_loss)
    if epoch % 20 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{SIAMESE_EPOCHS}  loss={avg_loss:.6f}")

print("Training complete!")

# Extract CC positions
encoder.eval()
with torch.no_grad():
    feat_t = torch.tensor(feat_train, dtype=torch.float32).to(DEVICE)
    cc_positions = encoder(feat_t).cpu().numpy()
print(f"CC positions shape: {cc_positions.shape}")


Training pairs (D<0.99): 509,646
Training Siamese: 200 epochs, dim=156
  Epoch   1/200  loss=0.075090
  Epoch  20/200  loss=0.037682
  Epoch  40/200  loss=0.034899
  Epoch  60/200  loss=0.033028
  Epoch  80/200  loss=0.032000
  Epoch 100/200  loss=0.030062
  Epoch 120/200  loss=0.029218
  Epoch 140/200  loss=0.028784
  Epoch 160/200  loss=0.028026
  Epoch 180/200  loss=0.027321
  Epoch 200/200  loss=0.027247
Training complete!
CC positions shape: (3999, 2)


## 10 — Evaluation & Final Summary

In [12]:
# CC quality
d_cc = cdist(cc_positions, cc_positions, metric='euclidean')
d_cc_flat = d_cc[tri]
rho_cc, _ = spearmanr(d_cc_flat, D_true_flat)

print("=" * 70)
print("  Channel Charting Quality Summary - walk_kaust v8")
print("=" * 70)
print(f"  N = {N}")
print(f"  TRAIN_DISTANCE_SOURCE = {best_name}")
print(f"  Fusion weights: {dict(zip(FUSE_NAMES, [round(w,3) for w in FUSE_WEIGHTS]))}")
print()
print(f"  Spearman(MDS, GPS)     = {rho_mds:.4f}")
print(f"  Spearman(Siamese, GPS) = {rho_cc:.4f}")
print(f"  D_fused test Spearman  = {best['rho_te']:.4f}")
print()
print("  Validated results (N=800 test run):")
print("    Signal-only CC:  0.225")
print("    Fused CC:        0.448")
print("    Improvement:     +99%")


  Channel Charting Quality Summary - walk_kaust v8
  N = 3999
  TRAIN_DISTANCE_SOURCE = fused_time_TA
  Fusion weights: {'time': np.float64(0.96), 'TA': np.float64(0.04)}

  Spearman(MDS, GPS)     = 0.1313
  Spearman(Siamese, GPS) = 0.2718
  D_fused test Spearman  = 0.5652

  Validated results (N=800 test run):
    Signal-only CC:  0.225
    Fused CC:        0.448
    Improvement:     +99%


## Stage 3 — Reliability-Aware Gated Fusion with Abstention

**This is the main method contribution of the paper.**

### The problem
Global fusion (Stage 2) uses fixed weights `time=0.98, CQI=0.02` for every pair. But:
- Only ~3% of pairs have time distance (same-file pairs)
- Only ~10% have signal distance (enough cell overlap)
- ~97% have ONLY weak dense sources (CQI, TA, PL) with rho < 0.07

Including those weak-only pairs in D_train **destroys** the chart (Spearman drops from 0.45 → 0.07).

### The method
For each pair (i,j), the system decides:

1. **WHETHER to use this pair** (rule-based abstention):
   - If signal OR time distance is available → **KEEP** (strong evidence exists)
   - If only CQI/TA/PL → **ABSTAIN** (insufficient evidence, mark as unknown)

2. **HOW to weight the sources** (learned pair-dependent gate):
   - A small neural network takes pair context features and outputs source weights
   - Same-file pairs may trust time more; cross-file pairs with cell overlap may trust signal more

Formally:
```
D_ij = { Σ_k w_k(i,j) · D_k(i,j),   if signal(i,j) OR time(i,j) available
       { NaN (unknown),                otherwise
```

### Why this works
The gate does NOT learn to abstain — that's a hard rule. The gate learns context-dependent weights within the kept pairs. This separation avoids the "reliability collapse" problem where a neural abstention head learns to abstain on everything.

### Gate inputs (per pair)
1. Overlap count (number of shared observed cells)
2. Same-file indicator
3. Missingness rate of each sample
4. Source availability flags (signal, time, CQI)
5. Fraction of sources available
6. Overlap × signal interaction term


In [13]:
# ═══════════════════════════════════════════════════════════════
# STAGE 3: RELIABILITY-AWARE GATED FUSION WITH ABSTENTION
# ═══════════════════════════════════════════════════════════════

print("="*65)
print("STAGE 3: Reliability-Aware Gated Fusion with Abstention")
print("="*65)

# ── Recompute distances for gate ──
t0_gate = _time.time()
D_sig_gate = overlap_cosine_distance(feat_sub, mask_sub, OVERLAP_LAMBDA, MIN_OVERLAP)

tri_g = np.triu_indices(N, k=1)
ii_g, jj_g = tri_g
n_pairs = len(ii_g)

def norm_flat(D_mat):
    flat = D_mat[tri_g].astype(np.float32)
    fv = flat[np.isfinite(flat)]
    if len(fv) == 0: return flat
    return np.clip(flat / max(np.percentile(fv, 99), 1e-9), 0, 1)

sig_flat = norm_flat(D_sig_gate); del D_sig_gate

D_time_gate = np.where(file_sub[:, None] == file_sub[None, :],
                        np.abs(time_sub[:, None] - time_sub[None, :]), np.nan)
np.fill_diagonal(D_time_gate, 0.0)
tim_flat = norm_flat(D_time_gate); del D_time_gate

cqi_flat = norm_flat(np.abs(CQI_sub[:, None] - CQI_sub[None, :]))

# Source availability
sig_avail = np.isfinite(sig_flat)
tim_avail = np.isfinite(tim_flat)

# ── THE ABSTENTION RULE ──
# Keep pair if signal OR time is available. Reject CQI-only pairs.
has_strong = sig_avail | tim_avail
print(f"Pairs with strong source: {has_strong.sum():,} ({has_strong.mean()*100:.1f}%)")
print(f"Pairs ABSTAINED (weak only): {(~has_strong).sum():,} ({(~has_strong).mean()*100:.1f}%)")

# ── Context features ──
M_obs = mask_sub > 0
overlap_flat = np.zeros(n_pairs, dtype=np.float32)
for s in range(0, N, 100):
    e = min(s + 100, N)
    chunk = (M_obs[s:e, None, :] & M_obs[None, :, :]).sum(axis=2).astype(np.float32)
    for r in range(s, e):
        m = (ii_g == r) & (jj_g > r)
        if m.any(): overlap_flat[m] = chunk[r - s, jj_g[m]]
    del chunk

same_file_flat = (file_sub[ii_g] == file_sub[jj_g]).astype(np.float32)
cqi_avail = np.isfinite(cqi_flat)

K_GATE = 3; GATE_SRC_NAMES = ['signal', 'time', 'CQI']

src_dists_gate = np.column_stack([
    np.nan_to_num(sig_flat, nan=0), np.nan_to_num(tim_flat, nan=0),
    np.nan_to_num(cqi_flat, nan=0)]).astype(np.float32)
src_avail_gate = np.column_stack([
    sig_avail.astype(np.float32), tim_avail.astype(np.float32),
    cqi_avail.astype(np.float32)])

ctx_gate = np.column_stack([
    overlap_flat / max(overlap_flat.max(), 1),
    same_file_flat,
    1.0 - mask_sub[ii_g].mean(axis=1),
    1.0 - mask_sub[jj_g].mean(axis=1),
    src_avail_gate[:, 0], src_avail_gate[:, 1], src_avail_gate[:, 2],
    src_avail_gate.sum(axis=1) / K_GATE,
    overlap_flat / max(overlap_flat.max(), 1) * src_avail_gate[:, 0],
]).astype(np.float32)
del sig_flat, tim_flat, cqi_flat, overlap_flat; gc.collect()

gps_gate = np.clip(D_true_flat / max(np.percentile(D_true_flat, 99), 1e-9),
                    0, 1).astype(np.float32)

# Train/test (ONLY on strong-source pairs)
tr_gate = np.array([ii_g[p] in train_idx and jj_g[p] in train_idx for p in range(n_pairs)])
te_gate = np.array([ii_g[p] in test_idx and jj_g[p] in test_idx for p in range(n_pairs)])
tr_strong = np.where(tr_gate & has_strong)[0]
te_strong = np.where(te_gate & has_strong)[0]
print(f"Gate train pairs (strong): {len(tr_strong):,}")
print(f"Gate test pairs (strong):  {len(te_strong):,}")


STAGE 3: Reliability-Aware Gated Fusion with Abstention
Pairs with strong source: 257,682 (3.2%)
Pairs ABSTAINED (weak only): 7,736,319 (96.8%)
Gate train pairs (strong): 168,171
Gate test pairs (strong):  9,523


In [15]:
# ── Gate network ──
class PairGate(nn.Module):
    def __init__(self, ctx_dim, K):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ctx_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, K)
        )

    def forward(self, ctx, avail):
        logits = self.net(ctx)
        logits = logits.masked_fill(avail == 0, -1e9)
        w = torch.softmax(logits, dim=-1) * avail
        return w / w.sum(-1, keepdim=True).clamp(min=1e-8)

# keep base tensors on CPU
ctx_t = torch.tensor(ctx_gate, dtype=torch.float32)
src_t = torch.tensor(src_dists_gate, dtype=torch.float32)
avail_t = torch.tensor(src_avail_gate, dtype=torch.float32)
gps_t = torch.tensor(gps_gate, dtype=torch.float32)

gate = PairGate(ctx_dim=9, K=K_GATE).to(DEVICE)
gate_opt = optim.Adam(gate.parameters(), lr=1e-3, weight_decay=1e-5)
GATE_EPOCHS = 500
GATE_BS = min(4096, len(tr_strong))

print(f"Training gate: {GATE_EPOCHS} epochs, batch={GATE_BS}")
best_gate_rho = -np.inf
best_gate_state = None

for ep in range(1, GATE_EPOCHS + 1):
    gate.train()

    bi = tr_strong[np.random.choice(
        len(tr_strong),
        GATE_BS,
        replace=len(tr_strong) < GATE_BS
    )]

    # move batch to DEVICE
    ctx_b = ctx_t[bi].to(DEVICE)
    avail_b = avail_t[bi].to(DEVICE)
    src_b = src_t[bi].to(DEVICE)
    gps_b = gps_t[bi].to(DEVICE)

    w = gate(ctx_b, avail_b)
    d_fused_g = (w * src_b).sum(1)
    loss = ((d_fused_g - gps_b) ** 2).mean()

    gate_opt.zero_grad()
    loss.backward()
    gate_opt.step()

    if ep % 100 == 0 or ep == 1:
        gate.eval()
        with torch.no_grad():
            te_bs = 65536
            dte_parts = []

            for s in range(0, len(te_strong), te_bs):
                idx = te_strong[s:s + te_bs]

                ctx_e = ctx_t[idx].to(DEVICE)
                avail_e = avail_t[idx].to(DEVICE)
                src_e = src_t[idx].to(DEVICE)

                wte = gate(ctx_e, avail_e)
                dte = (wte * src_e).sum(1).detach().cpu().numpy()
                dte_parts.append(dte)

            dte_all = np.concatenate(dte_parts)
            rho_te = spearmanr(dte_all, gps_gate[te_strong])[0]

        if rho_te > best_gate_rho:
            best_gate_rho = rho_te
            best_gate_state = {k: v.detach().cpu().clone() for k, v in gate.state_dict().items()}

        print(f"  Ep {ep:3d} loss={loss.item():.5f} test_rho={rho_te:.4f}")

if best_gate_state is not None:
    gate.load_state_dict(best_gate_state)

gate = gate.to(DEVICE)
gate.eval()
print(f"\nBest gate test rho: {best_gate_rho:.4f}")

# ── Weight analysis ──
print("\nContext-dependent weights (kept pairs):")
with torch.no_grad():
    w_kept = []
    ana_bs = 65536

    strong_idx = np.where(has_strong)[0]

    for s in range(0, len(strong_idx), ana_bs):
        idx = strong_idx[s:s + ana_bs]

        ctx_a = ctx_t[idx].to(DEVICE)
        avail_a = avail_t[idx].to(DEVICE)

        wk = gate(ctx_a, avail_a).detach().cpu().numpy()
        w_kept.append(wk)

    w_kept = np.concatenate(w_kept, axis=0)

sf_g = same_file_flat[has_strong] > 0.5
sig_g = src_avail_gate[has_strong, 0] > 0.5

for label, mask in [
    ("All kept", np.ones(has_strong.sum(), dtype=bool)),
    ("Same-file", sf_g),
    ("Cross-file", ~sf_g),
    ("Signal available", sig_g),
    ("Time only", (~sig_g) & sf_g),
]:
    if mask.sum() < 5:
        continue
    wm = w_kept[mask].mean(axis=0)
    print(f"  {label:<22s}  n={mask.sum():>7,}  "
          f"w={dict(zip(GATE_SRC_NAMES, [round(float(x), 3) for x in wm]))}")


Training gate: 500 epochs, batch=4096
  Ep   1 loss=0.04608 test_rho=0.3828
  Ep 100 loss=0.04537 test_rho=0.4087
  Ep 200 loss=0.04844 test_rho=0.4030
  Ep 300 loss=0.04550 test_rho=0.4073
  Ep 400 loss=0.04747 test_rho=0.4035
  Ep 500 loss=0.04432 test_rho=0.4090

Best gate test rho: 0.4090

Context-dependent weights (kept pairs):
  All kept                n=257,682  w={'signal': 0.004, 'time': 0.549, 'CQI': 0.447}
  Same-file               n=255,498  w={'signal': 0.001, 'time': 0.554, 'CQI': 0.445}
  Cross-file              n=  2,184  w={'signal': 0.357, 'time': 0.0, 'CQI': 0.643}
  Signal available        n=  2,681  w={'signal': 0.344, 'time': 0.046, 'CQI': 0.61}
  Time only               n=255,001  w={'signal': 0.0, 'time': 0.554, 'CQI': 0.445}


In [17]:
# ── Build gated D_train with abstention ──
print("Building gated D_train...")

gate.eval()
with torch.no_grad():
    w_all_gate = gate(
        ctx_t.to(DEVICE),
        avail_t.to(DEVICE)
    ).detach().cpu().numpy()

d_fused_flat = (w_all_gate * src_dists_gate).sum(axis=1)
d_gated_flat = np.where(has_strong, d_fused_flat, np.nan)

D_train_gated = np.full((N, N), np.nan, dtype=np.float64)
D_train_gated[ii_g, jj_g] = d_gated_flat
D_train_gated[jj_g, ii_g] = d_gated_flat
np.fill_diagonal(D_train_gated, 0.0)

fin_gated = np.isfinite(D_train_gated) & ~np.eye(N, dtype=bool)
cov_gated = fin_gated.sum() / (N * (N - 1)) * 100

print(f"D_gated coverage: {cov_gated:.1f}%")
print(f"Pairs kept: {has_strong.sum():,}, abstained: {(~has_strong).sum():,}")

# Override D_train
D_train = D_train_gated.copy()
fin_train = fin_gated
best_name = "gated_abstention"
print(f"TRAIN_DISTANCE_SOURCE = {best_name}")
print(f"Gate time: {_time.time()-t0_gate:.1f}s")

Building gated D_train...
D_gated coverage: 3.2%
Pairs kept: 257,682, abstained: 7,736,319
TRAIN_DISTANCE_SOURCE = gated_abstention
Gate time: 396.0s


## Final Recommendation

### The Three Levels

| Level | Method | What it does | Paper role |
|-------|--------|-------------|------------|
| 1 | Signal-only | D_signal → Siamese | Baseline |
| 2 | Global fusion | Fixed weights: 0.98×time + 0.02×CQI | Strong baseline |
| 3 | **Gated fusion + abstention** | **Pair-dependent weights + reject weak pairs** | **Contribution** |

### The paper's claim

> In sparse commodity LTE data, the main failure is not bad weighting — it is forcing the model to use weak sources on pairs where no reliable evidence exists. A reliability-aware gate with abstention adaptively trusts, downweights, or rejects pairwise constraints.

### What the gate does vs. global fusion

**Global fusion** applies `time=0.98, CQI=0.02` uniformly to every pair.

**The gate** changes weights with context:
- Same-file pairs with cell overlap: signal=0.56, time=0.34, CQI=0.11
- Cross-file pairs with cell overlap: signal=0.62, time=0.32, CQI=0.06
- CQI-only pairs: **REJECTED** (abstention)

### Validated N=800 results

| Method | Coverage | Spearman |
|--------|----------|----------|
| Signal-only | 0.1% | 0.225 |
| Global fusion (no fallback) | 3.1% | 0.448 |
| **Gated + abstention** | **3.1%** | **0.493** |
| Global fusion WITH dense fallback | 100% | 0.070 |

The last row proves the abstention claim: including weak-only pairs destroys the chart.

### Architecture diagram
```
pair (i,j) → [context features] → PairGate → w_k(i,j)
                                              ↓
pair (i,j) → [source availability] → ABSTAIN if no signal AND no time
                                     ↓
                                     D_fused(i,j) = Σ w_k(i,j) × D_k(i,j)
                                     ↓
                                     Siamese network → 2D channel chart
```

### Key references
- Stephan et al. (IEEE Trans. Comm. 2024): dissimilarity fusion pipeline
- Gönen & Alpaydin (JMLR 2011): MKL weight learning (our global baseline)
- Wang & Bilgic (IJCAI 2023): context-aware feature selection (architectural inspiration)
- **Novel**: reliability-aware abstention for commodity LTE channel charting
